# Synthetic Brain MRI Master Pipeline (Colab)

This notebook runs the full 5-step pipeline from a single place.

- Step 1: preprocessing (N4, skull-strip hook, registration hook, normalization)
- Step 2: working 2D DDPM baseline (training + sampling)
- Step 3: working 3D latent diffusion-style pipeline (autoencoder + latent denoising)
- Step 4: working validation suite (proxy FID, biomarker proxy, classification utility)
- Step 5: working benchmark suite (proposed model vs DCGAN/StyleGAN2/VAE3D-style baselines)

## 1) Mount Drive (Colab)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2) Configure Paths

In [ ]:
from pathlib import Path
import subprocess

DRIVE_ROOT = Path('/content/drive/MyDrive')

# Optional manual override if needed
REPO_DIR = None  # e.g., Path('/content/drive/MyDrive/your-folder/synthetic_brain_mri')
ADNI_COLAB_ROOT = None  # e.g., Path('/content/drive/MyDrive/AD_Research/ADNI1_Complete_1Yr_3T')
REPO_GIT_URL = 'https://github.com/SalmanSattar24/synthetic_brain_mri.git'

# User-provided local Windows path (reference)
ADNI_WINDOWS_PATH = r'C:\\All-Code\\AD_Research\\ADNI1_Complete_1Yr_3T'
RESEARCH_PLAN_LOCAL = r'C:\\All-Code\\AD_Research\\Files\\Synthetic_Brain_MRI_Generation__Research_Plan.pdf'

def _first_existing(candidates):
    for p in candidates:
        if p is not None and p.exists():
            return p
    return None

def _auto_find_repo():
    candidates = [
        DRIVE_ROOT / 'AD_Research' / 'synthetic_brain_mri',
        DRIVE_ROOT / 'synthetic_brain_mri',
        Path('/content/synthetic_brain_mri'),
        Path('/content') / 'drive' / 'MyDrive' / 'synthetic_brain_mri',
    ]
    found = _first_existing(candidates)
    if found is not None:
        return found

    # Fallback: search Drive for a folder named synthetic_brain_mri that has run_all_steps.py
    for p in DRIVE_ROOT.rglob('synthetic_brain_mri'):
        if p.is_dir() and (p / 'run_all_steps.py').exists():
            return p
    return None

def _ensure_repo_available(repo_dir):
    if repo_dir is not None and repo_dir.exists():
        return repo_dir

    clone_target = Path('/content/synthetic_brain_mri')
    if clone_target.exists() and (clone_target / 'run_all_steps.py').exists():
        return clone_target

    print('Repo not found on Drive. Cloning into /content ...')
    subprocess.run(['git', 'clone', REPO_GIT_URL, str(clone_target)], check=True)
    return clone_target

def _auto_find_adni_root():
    candidates = [
        DRIVE_ROOT / 'AD_Research' / 'ADNI1_Complete_1Yr_3T',
        DRIVE_ROOT / 'ADNI1_Complete_1Yr_3T',
        DRIVE_ROOT / 'AD_Research' / 'ADNI1_Complete_1Yr_3T' / 'ADNI',
        DRIVE_ROOT / 'ADNI1_Complete_1Yr_3T' / 'ADNI',
    ]
    found = _first_existing(candidates)
    if found is None:
        return None

    # Normalize to root folder that contains ADNI/
    return found.parent if found.name == 'ADNI' else found

if REPO_DIR is None:
    REPO_DIR = _auto_find_repo()
REPO_DIR = _ensure_repo_available(REPO_DIR)

if ADNI_COLAB_ROOT is None:
    ADNI_COLAB_ROOT = _auto_find_adni_root()

if ADNI_COLAB_ROOT is None:
    raise FileNotFoundError(
        'Could not find ADNI1_Complete_1Yr_3T in Google Drive. '
        'Set ADNI_COLAB_ROOT manually in this cell to your dataset root path.'
    )

ADNI_COLAB_ADNI_FOLDER = ADNI_COLAB_ROOT / 'ADNI'
if not ADNI_COLAB_ADNI_FOLDER.exists():
    raise FileNotFoundError(
        f'ADNI folder not found under dataset root: {ADNI_COLAB_ADNI_FOLDER}. '
        'Expected .../ADNI1_Complete_1Yr_3T/ADNI'
    )

if not (REPO_DIR / 'run_all_steps.py').exists():
    raise FileNotFoundError(
        f'run_all_steps.py not found in repo path: {REPO_DIR}. '
        'Please verify REPO_DIR points to synthetic_brain_mri.'
    )

print('Repo:', REPO_DIR)
print('ADNI dataset root:', ADNI_COLAB_ROOT)
print('ADNI folder:', ADNI_COLAB_ADNI_FOLDER)

FileNotFoundError: Could not find synthetic_brain_mri repo in Google Drive. Set REPO_DIR manually in this cell to your actual notebook repo path.

## 3) Install Dependencies

In [ ]:
import os
os.chdir(REPO_DIR)
!pip -q install -r requirements.txt

# Optional: install HD-BET separately if you want skull stripping active.
# !pip install hd-bet

In [ ]:
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. In Colab, set Runtime -> Change runtime type -> GPU.')

## 4) Patch Config for Colab and Run Pipeline

In [ ]:
import subprocess
import sys
import yaml

cfg_path = REPO_DIR / 'config' / 'config.yaml'
with cfg_path.open('r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['adni_root'] = str(ADNI_COLAB_ADNI_FOLDER)
cfg['paths']['data_raw'] = str(REPO_DIR / 'data' / 'raw')
cfg['paths']['data_interim'] = str(REPO_DIR / 'data' / 'interim')
cfg['paths']['data_processed'] = str(REPO_DIR / 'data' / 'processed')
cfg['paths']['step1_output'] = str(REPO_DIR / 'results' / 'step1')
cfg['step1']['registration']['mni_template_path'] = str(REPO_DIR / 'data' / 'templates' / 'MNI152_T1_1mm.nii')
cfg['step2']['conditional']['labels_csv'] = str(REPO_DIR / 'data' / 'metadata' / 'reference' / 'ADNIMERGE_03Apr2026.csv')
cfg['step3']['conditional']['labels_csv'] = str(REPO_DIR / 'data' / 'metadata' / 'reference' / 'ADNIMERGE_03Apr2026.csv')

# Full research-run defaults
cfg['step1']['smoke_test'] = False
cfg['step1']['num_workers'] = 4

tmp_cfg = REPO_DIR / 'config' / 'config.colab.yaml'
with tmp_cfg.open('w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

cmd = [sys.executable, 'run_all_steps.py', '--config', str(tmp_cfg), '--profile', 'config', '--preflight', '--resume-auto', '--start-step', '1', '--end-step', '5']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Pipeline run completed.')

## 5) Full Run Toggle

Notebook is now set to full-run defaults out of the box:
- `cfg['step1']['smoke_test'] = False`
- `--resume-auto` is enabled in the run command
- labels CSV and MNI template paths are patched to repo-relative Colab paths

If your Colab instance has lower RAM/CPU, reduce `cfg['step1']['num_workers']` to `2`.